# 🎢 Roller Coaster Dataset Analysis
Interactive visualizations built with Plotly. All charts are zoomable, hoverable, and filterable via the legend.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv('coasters_full_list_no_kings.csv')

# Normalize coaster_type labels for display
type_labels = {'steel': 'Steel', 'wood': 'Wood', 'launch': 'Launch', 'rmc': 'RMC (Steel-Wood Hybrid)'}
df['coaster_type_label'] = df['coaster_type'].map(type_labels).fillna(df['coaster_type'])

# Color palette consistent across charts
TYPE_COLORS = {
    'Steel':                  '#4C8BF5',
    'Wood':                   '#C8733A', 
    'Launch':                 '#2DC48D',
    'RMC (Steel-Wood Hybrid)':'#A855F7',
}

print(f"Loaded {len(df)} coasters across {df['park'].nunique()} parks in {df['state'].nunique()} states/provinces.")
df.head()

## 1 · Coaster Types — Count by Category

In [126]:
type_counts = (
    df.groupby('coaster_type_label')
      .size()
      .reset_index(name='count')
      .sort_values('count', ascending=False)
)

fig = px.bar(
    type_counts,
    x='coaster_type_label',
    y='count',
    color='coaster_type_label',
    color_discrete_map=TYPE_COLORS,
    text='count',
    labels={'coaster_type_label': 'Coaster Type', 'count': 'Number of Coasters'},
    title='Coasters by Type'
)
fig.update_traces(textposition='outside', hovertemplate='<b>%{x}</b><br>Count: %{y}<extra></extra>')
fig.update_layout(
    showlegend=False,
    xaxis_title=None,
    yaxis_title='Number of Coasters',
    plot_bgcolor='white',
    yaxis=dict(gridcolor='#eee'),
    margin=dict(t=60, b=40)
)
fig.show()

## 2 · Coasters per State / Province

In [127]:
state_type = (
    df.groupby(['state', 'coaster_type_label'])
      .size()
      .reset_index(name='count')
)

# Order states by total count
state_order = (
    state_type.groupby('state')['count'].sum()
               .sort_values(ascending=False)
               .index.tolist()
)

fig = px.bar(
    state_type,
    x='state',
    y='count',
    color='coaster_type_label',
    color_discrete_map=TYPE_COLORS,
    category_orders={'state': state_order},
    labels={'state': 'State / Province', 'count': 'Number of Coasters', 'coaster_type_label': 'Type'},
    title='Coasters per State / Province — Stacked by Type',
    barmode='stack',
    text_auto=True
)
fig.update_traces(hovertemplate='<b>%{x}</b> · %{fullData.name}<br>Count: %{y}<extra></extra>')
fig.update_layout(
    plot_bgcolor='white',
    yaxis=dict(gridcolor='#eee', title='Number of Coasters'),
    xaxis_title=None,
    legend_title='Coaster Type',
    margin=dict(t=60, b=40)
)
fig.show()

## 3 · Total Combined Drop Height if You Rode Every Coaster
Summed from `drop_ft`. Coasters with no recorded drop are noted.

In [128]:
drop_data = df.dropna(subset=['drop_ft']).copy()
missing_drop = df['drop_ft'].isna().sum()
total_drop_ft = drop_data['drop_ft'].sum()
total_drop_miles = total_drop_ft / 5280

# Breakdown by park for the waterfall
park_drop = (
    drop_data.groupby('park')['drop_ft']
             .sum()
             .sort_values(ascending=False)
             .reset_index()
)

print(f"Total combined drop (known coasters): {total_drop_ft:,.0f} ft  /  {total_drop_miles:.2f} miles")
print(f"({missing_drop} coasters have no recorded drop_ft and are excluded)")

fig = px.bar(
    park_drop,
    x='drop_ft',
    y='park',
    orientation='h',
    color='drop_ft',
    color_continuous_scale='Blues',
    text=park_drop['drop_ft'].apply(lambda x: f'{x:,.0f} ft'),
    labels={'drop_ft': 'Combined Drop (ft)', 'park': ''},
    title=f'Combined Drop Height per Park  |  Dataset Total: {total_drop_ft:,.0f} ft ({total_drop_miles:.2f} miles)'
)
fig.update_traces(
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Combined drop: %{x:,.0f} ft<extra></extra>'
)
fig.update_layout(
    coloraxis_showscale=False,
    plot_bgcolor='white',
    xaxis=dict(gridcolor='#eee', title='Combined Drop (ft)'),
    margin=dict(t=60, r=120, b=40)
)
fig.show()

Total combined drop (known coasters): 7,807 ft  /  1.48 miles
(35 coasters have no recorded drop_ft and are excluded)


## 4 · Total Track Length if You Rode Every Coaster
Uses actual published values from `track_length_ft` in the dataset. Coasters without reported length are listed as missing.

In [129]:
track_data = df.dropna(subset=['track_length_ft']).copy()
missing_track = df['track_length_ft'].isna().sum()
total_track_ft = track_data['track_length_ft'].sum()
total_track_mi = total_track_ft / 5280

print(f"Total track length (known coasters): {total_track_ft:,.0f} ft  /  {total_track_mi:.1f} miles")
print(f"({missing_track} coasters have no recorded track_length_ft and are excluded)")

park_track = (
    track_data.groupby('park')['track_length_ft']
              .sum()
              .sort_values(ascending=False)
              .reset_index()
)
park_track['track_mi'] = park_track['track_length_ft'] / 5280

fig = px.bar(
    park_track,
    x='track_length_ft',
    y='park',
    orientation='h',
    color='track_length_ft',
    color_continuous_scale='Greens',
    text=park_track['track_mi'].apply(lambda x: f'{x:.1f} mi'),
    labels={'track_length_ft': 'Total Track Length (ft)', 'park': ''},
    title=f'Actual Total Track Length per Park  |  Dataset Total: {total_track_mi:.1f} miles'
)
fig.update_traces(
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Total track: %{x:,.0f} ft<extra></extra>'
)
fig.update_layout(
    coloraxis_showscale=False,
    plot_bgcolor='white',
    xaxis=dict(gridcolor='#eee', title='Total Track Length (ft)'),
    margin=dict(t=60, r=100, b=40)
)
fig.show()

Total track length (known coasters): 222,008 ft  /  42.0 miles
(16 coasters have no recorded track_length_ft and are excluded)


## 5 · Coasters per Park

In [130]:
park_type = (
    df.groupby(['park', 'coaster_type_label'])
      .size()
      .reset_index(name='count')
)

park_order = (
    park_type.groupby('park')['count'].sum()
              .sort_values(ascending=False)
              .index.tolist()
)

fig = px.bar(
    park_type,
    x='count',
    y='park',
    color='coaster_type_label',
    color_discrete_map=TYPE_COLORS,
    orientation='h',
    category_orders={'park': park_order},
    barmode='stack',
    text_auto=True,
    labels={'count': 'Number of Coasters', 'park': '', 'coaster_type_label': 'Type'},
    title='Coasters per Park — Stacked by Type'
)
fig.update_traces(hovertemplate='<b>%{y}</b> · %{fullData.name}<br>Count: %{x}<extra></extra>')
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(gridcolor='#eee', title='Number of Coasters'),
    legend_title='Coaster Type',
    margin=dict(t=60, b=40)
)
fig.show()

## Bonus · Speed vs. Drop — Scatter by Park (bubble size = ride time)
Each dot is one coaster. Bubble size reflects ride duration in seconds. Hover for details.

In [131]:
scatter_df = df.dropna(subset=['max_speed_mph', 'drop_ft', 'ride_time_sec']).copy()

fig = px.scatter(
    scatter_df,
    y='drop_ft',
    x='max_speed_mph',
    color='park',
    size='ride_time_sec',
    size_max=30,
    hover_name='coaster',
    hover_data={'park': True, 'year_built': True, 'drop_ft': True,
                'max_speed_mph': True, 'ride_time_sec': True,
                'coaster_type_label': True},
    labels={
        'drop_ft': 'Drop (ft)',
        'max_speed_mph': 'Top Speed (mph)',
        'park': 'Park',
        'ride_time_sec': 'Ride Time (sec)',
        'coaster_type_label': 'Type'
    },
    title='Speed vs. Drop by Park  (bubble size = ride time)'
)
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(gridcolor='#eee'),
    yaxis=dict(gridcolor='#eee'),
    legend_title='Park',
    margin=dict(t=60, b=40)
)
fig.show()

## 6 · Family Ride Totals
Who has ridden the most coasters, where those rides happened, and who has accumulated the most total drop height by manufacturer.

In [ ]:
family_cols = ['Kobra Kid', 'Melanie', 'DooDad', 'Mom']

rides_long = (
    df.melt(
        id_vars=['park', 'manufacturer', 'drop_ft', 'coaster'],
        value_vars=family_cols,
        var_name='person',
        value_name='ridden'
    )
    .loc[lambda data: data['ridden'] == True]
    .copy()
)

person_order = (
    rides_long.groupby('person')['coaster']
    .count()
    .sort_values(ascending=False)
    .index.tolist()
)

park_counts = (
    rides_long.groupby(['person', 'park'])['coaster']
    .count()
    .reset_index(name='coaster_count')
)

fig = px.bar(
    park_counts,
    x='coaster_count',
    y='person',
    color='park',
    orientation='h',
    barmode='stack',
    category_orders={'person': person_order},
    text_auto=True,
    labels={'coaster_count': 'Coasters Ridden', 'person': '', 'park': 'Park'},
    title='Who Has Ridden the Most Coasters?  Broken Out by Park'
)
fig.update_traces(
    hovertemplate='<b>%{y}</b><br>%{fullData.name}: %{x} coasters<extra></extra>'
)
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(gridcolor='#eee', title='Coasters Ridden'),
    legend_title='Park',
    margin=dict(t=60, b=40)
)
fig.show()

drop_totals = (
    rides_long.dropna(subset=['drop_ft'])
    .groupby(['person', 'manufacturer'])['drop_ft']
    .sum()
    .reset_index()
)

fig = px.bar(
    drop_totals,
    x='drop_ft',
    y='person',
    color='manufacturer',
    orientation='h',
    barmode='stack',
    category_orders={'person': person_order},
    labels={'drop_ft': 'Total Drop Height (ft)', 'person': '', 'manufacturer': 'Manufacturer'},
    title='Total Drop Height Ridden by Person  Broken Out by Manufacturer'
)
fig.update_traces(
    hovertemplate='<b>%{y}</b><br>%{fullData.name}: %{x:,.0f} ft of drop<extra></extra>'
)
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(gridcolor='#eee', title='Total Drop Height (ft)'),
    legend_title='Manufacturer',
    margin=dict(t=60, b=40)
)
fig.show()